# Week 6 PII Masking and Data Sanitization Gateway

This notebook replays saved synthetic evidence without an API key. Run `evaluate.py` to generate fresh Presidio results.


In [1]:
from pathlib import Path
import json
root = Path.cwd()
evidence = json.loads((root / "outputs/evaluation.json").read_text(encoding="utf-8"))
assert evidence["synthetic_only"]
print(evidence["versions"])


{'presidio-analyzer': '2.2.366', 'presidio-anonymizer': '2.2.366', 'spacy': '3.8.18', 'en-core-web-sm': '3.8.0'}


## Recompute the reported metrics

Exact entity-and-span scoring excludes the four challenge cases. String removal is reported separately.


In [2]:
rows = evidence["cases"]
standard = [r for r in rows if r["group"] != "challenge"]
tp, fp, fn = [sum(r[k] for r in standard) for k in ("tp", "fp", "fn")]
precision, recall = tp / (tp + fp), tp / (tp + fn)
assert precision == evidence["metrics"]["exact_span_precision"]
assert recall == evidence["metrics"]["exact_span_recall"]
print({"precision": precision, "recall": recall, "tp": tp, "fp": fp, "fn": fn})


{'precision': 0.95, 'recall': 0.95, 'tp': 19, 'fp': 1, 'fn': 1}


In [3]:
for row in rows:
    if row["remaining_count"] or row["fp"] or row["fn"]:
        print(row["id"], row["group"], row["sanitized_text"], "remaining:", row["remaining_count"])


c10 standard My social security number is <PHONE_NUMBER>. remaining: 0
c25 challenge <PERSON> [at] example [dot] com. remaining: 0
c26 challenge Email alex＠example.com. remaining: 1
c27 challenge My name is <PERSON>. remaining: 0
c28 challenge My private account alias is blue-falcon-77. remaining: 1


## Verify downstream evidence

These are local HTTP test receivers, not a live LLM or vector database.


In [4]:
forwarding = evidence["http_forwarding"]
assert forwarding["statuses"] == [200, 200]
assert len(forwarding["received"]) == 2
for request in forwarding["received"]:
    assert "alex@example.com" not in json.dumps(request)
    assert "<EMAIL_ADDRESS>" in request["payload"]["text"]
print("Both local HTTP receivers received sanitized text only.")


Both local HTTP receivers received sanitized text only.


## Fresh runs

From PowerShell use `.\.venv\Scripts\python.exe evaluate.py`. The Streamlit demo runs with `run.ps1`. Sensitive real inputs must not be added to this notebook or GitHub.
